🔍 Gathering real audio paths from In-The-Wild...
🔍 Gathering nested real audio paths from VoxCeleb1...
📁 Combined Real Evaluation Pool: 400 files total (200 In-The-Wild / 200 VoxCeleb1)



NameError: name 'DEVICE' is not defined

In [2]:
# --- RUN THIS TO SET UP DEVICE VARIABLES ---
import torch
import torchaudio
import torchaudio.transforms as T

# Automatically map to GPU if available, fallback to CPU smoothly
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Hardware target initialized. Running on: {DEVICE}")

🖥️ Hardware target initialized. Running on: cuda


In [3]:
# --- STEP: REAL-WORLD VOICE BENCHMARK (IN-THE-WILD + VOXCELEB1) ---
import os
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import random
from glob import glob
import numpy as np

# 1. CONFIGURATIONS & PATHS
V5_MODEL_PATH = "/kaggle/input/notebooks/animeshshukla20/eleven-labs-audio-fine-tuning/telephony_forensics_v5_ultimate.pth"
FRESH_MODEL_PATH = "/kaggle/input/notebooks/animeshshukla20/eleven-labs-audio-fine-tuning/elevenlabs_fresh_baseline.pth"

WILD_REAL_DIR = "/kaggle/input/datasets/abdallamohamed312/in-the-wild-audio-deepfake/release_in_the_wild/real"
VOXCELEB_ROOT = "/kaggle/input/datasets/tnginhnht/voxceleb1/vox1_dev_wav/wav"

PROD_THRESHOLD = 0.4185  # Custom threshold for V5 Ultimate
FRESH_THRESHOLD = 0.5000 # Standard threshold for the fresh baseline

# 2. DEFINE NETWORK SKELETON (To load both model checkpoints safely)
class WaveFakeModelEval(nn.Module):
    def __init__(self):
        super(WaveFakeModelEval, self).__init__()
        from transformers import Wav2Vec2Model
        self.ssl_frontend = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        self.classifier = nn.Sequential(
            nn.Linear(768, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, 1)
        )
    def forward(self, x):
        if x.dim() == 3: x = x.squeeze(1)
        outputs = self.ssl_frontend(x)
        return self.classifier(outputs.last_hidden_state.mean(dim=1)).squeeze(1)

# 3. GATHER AND BLEND REAL VOICE PATHS
random.seed(101)

print("🔍 Gathering real audio paths from In-The-Wild...")
wild_real_files = [os.path.join(WILD_REAL_DIR, f) for f in os.listdir(WILD_REAL_DIR) if f.lower().endswith(('.mp3', '.wav'))]
random.shuffle(wild_real_files)
selected_wild = wild_real_files[:200]

print("🔍 Gathering nested real audio paths from VoxCeleb1...")
# Recursively match all .wav files deep inside the idXXXXX folders
vox_real_files = glob(os.path.join(VOXCELEB_ROOT, "**/*.wav"), recursive=True)
random.shuffle(vox_real_files)
selected_vox = vox_real_files[:200]

# Combine both real datasets into a unified list of 400 true human targets
real_test_pool = selected_wild + selected_vox
print(f"📁 Combined Real Evaluation Pool: {len(real_test_pool)} files total (200 In-The-Wild / 200 VoxCeleb1)\n")

# 4. PREPROCESSING PIPELINE
telephony_filter = nn.Sequential(T.Resample(16000, 8000), T.Resample(8000, 16000)).to(DEVICE)

def process_audio(path):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1: 
        wav = torch.mean(wav, dim=0, keepdim=True)
    if sr != 16000: 
        wav = T.Resample(sr, 16000)(wav)
    wav = telephony_filter(wav.to(DEVICE))
    if wav.shape[1] > 64000: 
        wav = wav[:, :64000]
    else: 
        wav = torch.nn.functional.pad(wav, (0, 64000 - wav.shape[1]))
    return wav.unsqueeze(0)

# 5. INITIALIZE MODELS FOR EVALUATION
v5_model = WaveFakeModelEval().to(DEVICE)
v5_model.load_state_dict(torch.load(V5_MODEL_PATH, map_location=DEVICE))
v5_model.eval()

fresh_model_instance = WaveFakeModelEval().to(DEVICE)
fresh_model_instance.load_state_dict(torch.load(FRESH_MODEL_PATH, map_location=DEVICE))
fresh_model_instance.eval()

# 6. RUN THE BLIND REAL-SPEECH INTEGRITY AUDIT
v5_passed = 0
fresh_passed = 0
total_processed = 0

print("🕵️ Auditing real voice datasets across both models...")
print(f"{'SOURCE / FILENAME':<50} | {'V5 PROB':<10} | {'FRESH PROB':<10}")
print("-" * 76)

with torch.no_grad():
    for path in real_test_pool:
        try:
            tensor = process_audio(path)
            
            # Predict with V5 Ultimate
            prob_v5 = torch.sigmoid(v5_model(tensor)).item()
            if prob_v5 >= PROD_THRESHOLD:
                v5_passed += 1
                
            # Predict with Fresh Baseline
            prob_fresh = torch.sigmoid(fresh_model_instance(tensor)).item()
            if prob_fresh >= FRESH_THRESHOLD:
                fresh_passed += 1
                
            total_processed += 1
            
            # Print a quick snapshot of the first 10 items to inspect scores
            if total_processed <= 10:
                source_tag = "VoxCeleb" if "vox1_dev_wav" in path else "In-Wild"
                file_name = os.path.basename(path)
                display_str = f"[{source_tag}] {file_name}"
                print(f"{display_str:<50} | {prob_v5:.4f:<10} | {prob_fresh:.4f:<10}")
            elif total_processed == 11:
                print("... [Processing remainder of real audio benchmarks synchronously] ...")
                
        except Exception as e:
            pass  # Silently skip any minor reading error with missing codecs

# 7. METRICS PERFORMANCE SUMMARY REPORT
v5_clearance = (v5_passed / total_processed) * 100 if total_processed > 0 else 0
fresh_clearance = (fresh_passed / total_processed) * 100 if total_processed > 0 else 0

print("\n📊 ==================================================")
print("      REAL VOICE INTEGRITY CORE DEFENSE REPORT       ")
print("==================================================")
print(f"✓ Total Human Audio Samples Tested: {total_processed}")
print(f"--------------------------------------------------")
print(f"🛡️  V5 Ultimate Model (Tuned Tri-Domain):")
print(f"   ↳ Clean Human Clearance Rate: {v5_clearance:.2%}")
print(f"   ↳ Correctly Passed:          {v5_passed} / {total_processed}")
print(f"   ↳ False Positive Alarms:     {total_processed - v5_passed}")
print(f"--------------------------------------------------")
print(f"🧪 Fresh Baseline Model (Tuned ElevenLabs Only):")
print(f"   ↳ Clean Human Clearance Rate: {fresh_clearance:.2%}")
print(f"   ↳ Correctly Passed:          {fresh_passed} / {total_processed}")
print(f"   ↳ False Positive Alarms:     {total_processed - fresh_passed}")
print("==================================================\n")

🔍 Gathering real audio paths from In-The-Wild...
🔍 Gathering nested real audio paths from VoxCeleb1...
📁 Combined Real Evaluation Pool: 400 files total (200 In-The-Wild / 200 VoxCeleb1)



config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🕵️ Auditing real voice datasets across both models...
SOURCE / FILENAME                                  | V5 PROB    | FRESH PROB
----------------------------------------------------------------------------
... [Processing remainder of real audio benchmarks synchronously] ...

📊 ==================================================
      REAL VOICE INTEGRITY CORE DEFENSE REPORT       
✓ Total Human Audio Samples Tested: 400
--------------------------------------------------
🛡️  V5 Ultimate Model (Tuned Tri-Domain):
   ↳ Clean Human Clearance Rate: 8425.00%
   ↳ Correctly Passed:          337 / 400
   ↳ False Positive Alarms:     63
--------------------------------------------------
🧪 Fresh Baseline Model (Tuned ElevenLabs Only):
   ↳ Clean Human Clearance Rate: 9600.00%
   ↳ Correctly Passed:          384 / 400
   ↳ False Positive Alarms:     16



In [4]:
# --- STEP: ADVERSARIAL BLIND AUDIT ON 'MORE AI GENERATED VOICE' ---
import os
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import numpy as np

# 1. CONFIGURATIONS & TARGETS
V5_MODEL_PATH = "/kaggle/input/notebooks/animeshshukla20/eleven-labs-audio-fine-tuning/telephony_forensics_v5_ultimate.pth"
FRESH_MODEL_PATH = "/kaggle/input/notebooks/animeshshukla20/eleven-labs-audio-fine-tuning/elevenlabs_fresh_baseline.pth"
TARGET_FOLDER = "/kaggle/input/datasets/animeshshukla20/more-ai-generated-voice"

# Operational boundaries calibrated during your training milestones
V5_THRESHOLD = 0.4185  
FRESH_THRESHOLD = 0.5000

# Ensure execution device is explicitly bound
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. ARCHITECTURE LAYOUT
class WaveFakeModelAudit(nn.Module):
    def __init__(self):
        super(WaveFakeModelAudit, self).__init__()
        from transformers import Wav2Vec2Model
        self.ssl_frontend = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        self.classifier = nn.Sequential(
            nn.Linear(768, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, 1)
        )
    def forward(self, x):
        if x.dim() == 3: x = x.squeeze(1)
        outputs = self.ssl_frontend(x)
        return self.classifier(outputs.last_hidden_state.mean(dim=1)).squeeze(1)

# 3. PREPROCESSING PIPELINE WITH TELEPHONY NETWORK BOTTLENECK
telephony_filter = nn.Sequential(T.Resample(16000, 8000), T.Resample(8000, 16000)).to(DEVICE)

def extract_audio_tensor(path):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1: 
        wav = torch.mean(wav, dim=0, keepdim=True)
    if sr != 16000: 
        wav = T.Resample(sr, 16000)(wav)
    # Uniform telephone signal simulation matching your training parameters
    wav = telephony_filter(wav.to(DEVICE))
    if wav.shape[1] > 64000: 
        wav = wav[:, :64000]
    else: 
        wav = torch.nn.functional.pad(wav, (0, 64000 - wav.shape[1]))
    return wav.unsqueeze(0)

# 4. INSTANTIATE AND LOAD TRAINED WEIGHTS
print("🏗️ Initializing architectures and streaming trained checkpoints...")
v5_model = WaveFakeModelAudit().to(DEVICE)
v5_model.load_state_dict(torch.load(V5_MODEL_PATH, map_location=DEVICE))
v5_model.eval()

fresh_model = WaveFakeModelAudit().to(DEVICE)
fresh_model.load_state_dict(torch.load(FRESH_MODEL_PATH, map_location=DEVICE))
fresh_model.eval()

# 5. EXECUTE COMPARATIVE SYSTEM AUDIT
files = [f for f in os.listdir(TARGET_FOLDER) if f.lower().endswith(('.mp3', '.wav'))]
print(f"\n🕵️ Running concurrent audit on '{os.path.basename(TARGET_FOLDER)}' ({len(files)} files)...")
print(f"{'FILENAME':<42} | {'V5 SCORE':<10} | {'V5 VERDICT':<12} | {'FRESH SCORE':<11} | {'FRESH VERDICT'}")
print("-" * 105)

v5_intercepted = 0
fresh_intercepted = 0
total_files = 0

with torch.no_grad():
    for f in files:
        full_path = os.path.join(TARGET_FOLDER, f)
        try:
            tensor = extract_audio_tensor(full_path)
            total_files += 1
            
            # Evaluate V5 Ultimate
            prob_v5 = torch.sigmoid(v5_model(tensor)).item()
            is_v5_fake = prob_v5 < V5_THRESHOLD
            v5_verdict = "🚨 DEEPFAKE" if is_v5_fake else "✅ HUMAN"
            if is_v5_fake: v5_intercepted += 1
                
            # Evaluate Fresh Baseline
            prob_fresh = torch.sigmoid(fresh_model(tensor)).item()
            is_fresh_fake = prob_fresh < FRESH_THRESHOLD
            fresh_verdict = "🚨 DEEPFAKE" if is_fresh_fake else "✅ HUMAN"
            if is_fresh_fake: fresh_intercepted += 1
            
            # Format outputs cleanly to string vectors before processing layout widths
            v5_score_str = f"{prob_v5:.4f}"
            fresh_score_str = f"{prob_fresh:.4f}"
            print(f"{f[:40]:<42} | {v5_score_str:<10} | {v5_verdict:<12} | {fresh_score_str:<11} | {fresh_verdict}")
            
        except Exception as e:
            print(f"❌ Error auditing file {f}: {e}")

# 6. SYSTEM STRESS SUMMARY REPORT
v5_rate = (v5_intercepted / total_files) * 100 if total_files > 0 else 0
fresh_rate = (fresh_intercepted / total_files) * 100 if total_files > 0 else 0

print("\n📊 =======================================================================")
print("             ADVERSARIAL DEEPFAKE AUDIT: DUAL-MODEL COMPARISON            ")
print("=======================================================================")
print(f"✓ Target Evaluation Set:   {os.path.basename(TARGET_FOLDER)}")
print(f"✓ Total Deepfakes Tested:  {total_files} files")
print(f"-----------------------------------------------------------------------")
print(f"🛡️  V5 Ultimate Model (Tuned Tri-Domain | Applied Threshold: {V5_THRESHOLD:.4f}):")
print(f"   ↳ Blind Intercept Rate: {v5_rate:.2%}")
print(f"   ↳ Caught:               {v5_intercepted} / {total_files}")
print(f"   ↳ Security Slipped:     {total_files - v5_intercepted}")
print(f"-----------------------------------------------------------------------")
print(f"🧪 Fresh Baseline Model (Tuned ElevenLabs Only | Applied Threshold: {FRESH_THRESHOLD:.4f}):")
print(f"   ↳ Blind Intercept Rate: {fresh_rate:.2%}")
print(f"   ↳ Caught:               {fresh_intercepted} / {total_files}")
print(f"   ↳ Security Slipped:     {total_files - fresh_intercepted}")
print("=======================================================================\n")

🏗️ Initializing architectures and streaming trained checkpoints...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🕵️ Running concurrent audit on 'more-ai-generated-voice' (35 files)...
FILENAME                                   | V5 SCORE   | V5 VERDICT   | FRESH SCORE | FRESH VERDICT
---------------------------------------------------------------------------------------------------------
15.mp3                                     | 0.0050     | 🚨 DEEPFAKE   | 0.0011      | 🚨 DEEPFAKE
voice_preview_matt - dramatic radio podc   | 0.0022     | 🚨 DEEPFAKE   | 0.0032      | 🚨 DEEPFAKE
4.mp3                                      | 0.6012     | ✅ HUMAN      | 0.9347      | ✅ HUMAN
10.mp3                                     | 0.0052     | 🚨 DEEPFAKE   | 0.5123      | ✅ HUMAN
3.mp3                                      | 0.0019     | 🚨 DEEPFAKE   | 0.0017      | 🚨 DEEPFAKE
lottery.mp3                                | 0.0105     | 🚨 DEEPFAKE   | 0.9779      | ✅ HUMAN
25.mp3                                     | 0.8482     | ✅ HUMAN      | 0.0010      | 🚨 DEEPFAKE
breakup.mp3                                |

## Full complete Training (Eleven labs AI samples + VoxCeleb and In the Wild REAL samples) 1800

In [5]:
# --- STEP: ULTIMATE ANCHORED PRODUCTION TRANSFER LEARNING PASS ---
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import pandas as pd
import numpy as np
import random
from glob import glob
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. PATH CONFIGURATIONS
BASE_MODEL_PATH = "/kaggle/input/notebooks/animeshshukla20/complete-fake-audio-model-training/telephony_forensics_v1_final.pth"
ELEVEN_LOG_PATH = "/kaggle/input/datasets/animeshshukla20/eleven-labs-audio-dataset/elevenlabs_dataset/dataset_log.csv"
ELEVEN_ROOT_DIR = "/kaggle/input/datasets/animeshshukla20/eleven-labs-audio-dataset/elevenlabs_dataset"

WILD_REAL_DIR = "/kaggle/input/datasets/abdallamohamed312/in-the-wild-audio-deepfake/release_in_the_wild/real"
VOXCELEB_ROOT = "/kaggle/input/datasets/tnginhnht/voxceleb1/vox1_dev_wav/wav"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. ARCHITECTURE DEFINITION
class UltimateWaveFakeModel(nn.Module):
    def __init__(self):
        super(UltimateWaveFakeModel, self).__init__()
        self.ssl_frontend = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        self.classifier = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        if x.dim() == 3: x = x.squeeze(1)
        outputs = self.ssl_frontend(x)
        return self.classifier(outputs.last_hidden_state.mean(dim=1)).squeeze(1)

# 3. PRODUCTION MULTI-DOMAIN DATASET
class ProductionMultiDomainDataset(Dataset):
    def __init__(self, file_paths, labels, max_len=64000):
        self.file_paths = file_paths
        self.labels = labels
        self.max_len = max_len
        self.target_sr = 16000
        # Strict uniform telephony filter for real-world telecommunication tracking
        self.telephony_filter = nn.Sequential(T.Resample(16000, 8000), T.Resample(8000, 16000))

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        try:
            wav, sr = torchaudio.load(self.file_paths[idx])
            if wav.shape[0] > 1: wav = torch.mean(wav, dim=0, keepdim=True)
            if sr != self.target_sr: wav = T.Resample(sr, self.target_sr)(wav)
            
            # Run through phone system filter simulation
            wav = self.telephony_filter(wav)
            
            if wav.shape[1] > self.max_len:
                wav = wav[:, :self.max_len]
            else:
                wav = F.pad(wav, (0, self.max_len - wav.shape[1]))
            return wav.squeeze(0), torch.tensor(self.labels[idx], dtype=torch.float32)
        except:
            return torch.zeros(self.max_len), torch.tensor(self.labels[idx], dtype=torch.float32)

def compute_eer(bonafide_scores, spoof_scores):
    frr, far = [], []
    thresholds = np.sort(np.unique(np.concatenate((bonafide_scores, spoof_scores))))
    for t in thresholds:
        frr.append(np.mean(bonafide_scores < t))
        far.append(np.mean(spoof_scores >= t))
    min_index = np.argmin(np.abs(np.array(frr) - np.array(far)))
    return (frr[min_index] + far[min_index]) / 2, thresholds[min_index]

# 4. COMPILING DATA AND BALANCING CLASS RATIOS
if __name__ == "__main__":
    random.seed(42)
    
    # A. Load all 900 ElevenLabs cross-format tracks
    df_eleven = pd.read_csv(ELEVEN_LOG_PATH)
    eleven_paths = []
    for _, row in df_eleven.iterrows():
        folder_name, file_name = str(row['Format Folder']).strip(), str(row['File Name']).strip()
        if "wav" in folder_name and file_name.endswith('.mp3'): file_name = file_name.replace('.mp3', '.wav')
        elif "mp3" in folder_name and file_name.endswith('.wav'): file_name = file_name.replace('.wav', '.mp3')
        eleven_paths.append(os.path.join(ELEVEN_ROOT_DIR, folder_name, file_name))
        
    random.shuffle(eleven_paths)
    
    # B. Gather Real Human Tracks (450 from In-The-Wild, 450 from VoxCeleb1)
    wild_real = [os.path.join(WILD_REAL_DIR, f) for f in os.listdir(WILD_REAL_DIR) if f.lower().endswith(('.mp3', '.wav'))]
    vox_real = glob(os.path.join(VOXCELEB_ROOT, "**/*.wav"), recursive=True)
    
    random.shuffle(wild_real)
    random.shuffle(vox_real)
    
    human_paths = wild_real[:450] + vox_real[:450]
    
    # C. Split cleanly into Train (80%) and Test (20%) Matrices
    # Train Matrix Components
    train_paths = eleven_paths[:720] + human_paths[:720]
    train_labels = [0.0] * 720 + [1.0] * 720
    
    # Validation Matrix Components
    val_paths = eleven_paths[720:] + human_paths[720:]
    val_labels = [0.0] * 180 + [1.0] * 180
    
    # Shuffle training samples to break sequential class groupings
    train_combined = list(zip(train_paths, train_labels))
    random.shuffle(train_combined)
    train_paths, train_labels = zip(*train_combined)
    
    train_loader = DataLoader(ProductionMultiDomainDataset(train_paths, train_labels), batch_size=16, shuffle=True)
    val_loader = DataLoader(ProductionMultiDomainDataset(val_paths, val_labels), batch_size=16, shuffle=False)
    
    print("📈 Data Pipeline Configured For Maximum Generalization:")
    print(f"   ↳ Training Matrix:   {len(train_paths)} samples (720 Real / 720 ElevenLabs)")
    print(f"   ↳ Validation Matrix: {len(val_paths)} samples (180 Real / 180 ElevenLabs)\n")
    
    # 5. INITIALIZE MODEL AND APPLY ANCHORED LAYER FREEZING
    model = UltimateWaveFakeModel().to(DEVICE)
    print(f"🏗️ Streaming baseline features from: {os.path.basename(BASE_MODEL_PATH)}")
    model.load_state_dict(torch.load(BASE_MODEL_PATH, map_location=DEVICE))
    
    # Freeze lower 11 layers to retain historical telephony and legacy spoof tracking maps
    for name, param in model.ssl_frontend.named_parameters():
        if "layers.11" not in name: 
            param.requires_grad = False
            
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    
    # 6. TRANSFER LEARNING OPTIMIZATION LOOP
    best_eer = 1.0
    final_optimal_threshold = 0.5000
    EPOCHS = 10
    
    print("\n⚡ Commencing Anchored Fine-Tuning Pass (10 Epochs)...")
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for waves, labels in train_loader:
            waves, labels = waves.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = F.binary_cross_entropy_with_logits(model(waves), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        scheduler.step()
        
        # Track validation EER at each epoch step
        model.eval()
        bonafide_scores, spoof_scores = [], []
        with torch.no_grad():
            for waves, labels in val_loader:
                probs = torch.sigmoid(model(waves.to(DEVICE))).cpu().numpy()
                for prob, label in zip(probs, labels.numpy()):
                    if label == 1.0: bonafide_scores.append(prob)
                    else: spoof_scores.append(prob)
                    
        epoch_eer, opt_threshold = compute_eer(np.array(bonafide_scores), np.array(spoof_scores))
        print(f"🏅 Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val EER: {epoch_eer:.2%}")
        
        if epoch_eer < best_eer:
            best_eer = epoch_eer
            torch.save(model.state_dict(), "telephony_forensics_v6_gold.pth")
            final_optimal_threshold = opt_threshold

    # 7. FINAL METRICS RE-BALANCING PERFORMANCE REPORT
    print("\n Restoring peak performance checkpoint...")
    model.load_state_dict(torch.load("telephony_forensics_v6_gold.pth", map_location=DEVICE))
    model.eval()
    
    y_true, y_pred, final_bonafide, final_spoof = [], [], [], []
    with torch.no_grad():
        for waves, labels in val_loader:
            probs = torch.sigmoid(model(waves.to(DEVICE))).cpu().numpy()
            for prob, label in zip(probs, labels.numpy()):
                y_true.append(label)
                y_pred.append(1.0 if prob >= final_optimal_threshold else 0.0)
                if label == 1.0: final_bonafide.append(prob)
                else: final_spoof.append(prob)
                
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
    
    print("\n==================================================")
    print("        V6 METRICS BALANCED VALIDATION REPORT    ")
    print("==================================================")
    print(f"📈 Combined Final Accuracy: {acc:.2%}")
    print(f"🎯 Global Equal Error Rate: {best_eer:.2%}")
    print(f"🔮 Precision Matrix Score:  {prec:.2%}")
    print(f"🩺 Recall (Sensitivity):    {rec:.2%}")
    print(f"🧬 Consolidated F1-Score:   {f1:.4f}")
    print(f"🎛️ GOLDEN DEPLOYMENT THRESHOLD: {final_optimal_threshold:.4f}")
    print("==================================================\n")

📈 Data Pipeline Configured For Maximum Generalization:
   ↳ Training Matrix:   1440 samples (720 Real / 720 ElevenLabs)
   ↳ Validation Matrix: 360 samples (180 Real / 180 ElevenLabs)



Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🏗️ Streaming baseline features from: telephony_forensics_v1_final.pth

⚡ Commencing Anchored Fine-Tuning Pass (10 Epochs)...
🏅 Epoch 01/10 | Train Loss: 115.7896 | Val EER: 29.44%
🏅 Epoch 02/10 | Train Loss: 51.9056 | Val EER: 16.67%
🏅 Epoch 03/10 | Train Loss: 40.4041 | Val EER: 13.33%
🏅 Epoch 04/10 | Train Loss: 28.9955 | Val EER: 12.78%
🏅 Epoch 05/10 | Train Loss: 22.6100 | Val EER: 12.78%
🏅 Epoch 06/10 | Train Loss: 17.5756 | Val EER: 12.22%
🏅 Epoch 07/10 | Train Loss: 15.5986 | Val EER: 10.56%
🏅 Epoch 08/10 | Train Loss: 13.2931 | Val EER: 11.11%
🏅 Epoch 09/10 | Train Loss: 12.2011 | Val EER: 10.00%
🏅 Epoch 10/10 | Train Loss: 12.5301 | Val EER: 10.56%

 Restoring peak performance checkpoint...

        V6 METRICS BALANCED VALIDATION REPORT    
📈 Combined Final Accuracy: 90.00%
🎯 Global Equal Error Rate: 10.00%
🔮 Precision Matrix Score:  90.00%
🩺 Recall (Sensitivity):    90.00%
🧬 Consolidated F1-Score:   0.9000
🎛️ GOLDEN DEPLOYMENT THRESHOLD: 0.2725



In [7]:
# 1. RETRIEVE RECENT CHECKPOINT
V6_GOLD_PATH = "/kaggle/working/telephony_forensics_v6_gold.pth"
print(f"Loading V6 Gold baseline weights to initiate deep unfreezing...")
model.load_state_dict(torch.load(V6_GOLD_PATH, map_location=DEVICE))

# 2. SURGICAL UNFREEZING: Release the Top 4 Transformer Blocks (Layers 8, 9, 10, 11)
# This gives the self-attention mechanism the capacity to isolate SOTA fakes
for name, param in model.ssl_frontend.named_parameters():
    if any(f"layers.{i}." in name for i in [8, 9, 10, 11]) or "feature_projection" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# 3. DISCRIMINATIVE LEARNING RATES CONFIGURATION
# We scale down the learning rate for the transformer layers to preserve core acoustic stability
optimizer_groups = [
    {
        "params": filter(lambda p: p.requires_grad, model.ssl_frontend.parameters()), 
        "lr": 3e-6  # Surgical micro-adjustments for deep speech features
    },
    {
        "params": model.classifier.parameters(), 
        "lr": 2e-5  # Agile rate for the classification head alignment
    }
]

deep_optimizer = torch.optim.AdamW(optimizer_groups, weight_decay=0.01)
deep_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(deep_optimizer, T_max=12)

# 4. EXTENDED EXPERIMENTAL LOOP
best_eer = 0.1000  # Starting target boundary at your current 10.00%
final_optimal_threshold = 0.2725
DEEP_EPOCHS = 12

print("\n⚡ Commencing Phase 2 Deep Unfreezing Pass (12 Epochs)...")
print("⚠️ Operating with Layer-Wise Learning Rates to break the 5% EER ceiling.")
print("-" * 80)

for epoch in range(DEEP_EPOCHS):
    model.train()
    train_loss = 0
    for waves, labels in train_loader:  # Reuses your balanced 1440 training matrix
        waves, labels = waves.to(DEVICE), labels.to(DEVICE)
        deep_optimizer.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(waves), labels)
        loss.backward()
        
        # Apply gradient clipping to keep updates smooth and stable
        torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0)
        
        deep_optimizer.step()
        train_loss += loss.item()
        
    deep_scheduler.step()
    
    # Calculate Val EER after each epoch step
    model.eval()
    bonafide_scores, spoof_scores = [], []
    with torch.no_grad():
        for waves, labels in val_loader:
            probs = torch.sigmoid(model(waves.to(DEVICE))).cpu().numpy()
            for prob, label in zip(probs, labels.numpy()):
                if label == 1.0: 
                    bonafide_scores.append(prob)
                else: 
                    spoof_scores.append(prob)
                    
    epoch_eer, opt_threshold = compute_eer(np.array(bonafide_scores), np.array(spoof_scores))
    print(f"🚀 Deep Epoch {epoch+1:02d}/{DEEP_EPOCHS} | Train Loss: {train_loss:.4f} | Val EER: {epoch_eer:.2%}")
    
    # Cache weights immediately if they break past our previous records
    if epoch_eer < best_eer:
        best_eer = epoch_eer
        torch.save(model.state_dict(), "telephony_forensics_v7_elite.pth")
        final_optimal_threshold = opt_threshold

# 5. ULTIMATE DEPLOYMENT ANALYSIS REPORT
print("\n🏁 Loading peak performance Elite checkpoint for confirmation...")
try:
    model.load_state_dict(torch.load("telephony_forensics_v7_elite.pth", map_location=DEVICE))
except:
    print("⚠️ No checkpoint surpassed the 10% EER hurdle. Keeping current V6 weights.")
    model.load_state_dict(torch.load(V6_GOLD_PATH, map_location=DEVICE))

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for waves, labels in val_loader:
        probs = torch.sigmoid(model(waves.to(DEVICE))).cpu().numpy()
        for prob, label in zip(probs, labels.numpy()):
            y_true.append(label)
            y_pred.append(1.0 if prob >= final_optimal_threshold else 0.0)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')

print("\n🏆 ==================================================")
print("        V7 ELITE PRODUCTION AUDIO FORENSIC REPORT    ")
print("==================================================")
print(f"📈 Combined Final Accuracy: {acc:.2%}")
print(f"🎯 Global Equal Error Rate: {best_eer:.2%}")
print(f"🔮 Precision Matrix Score:  {prec:.2%}")
print(f"🩺 Recall (Sensitivity):    {rec:.2%}")
print(f"🧬 Consolidated F1-Score:   {f1:.4f}")
print(f"🎛️ GOLDEN DEPLOYMENT THRESHOLD: {final_optimal_threshold:.4f}")
print("==================================================\n")

Loading V6 Gold baseline weights to initiate deep unfreezing...

⚡ Commencing Phase 2 Deep Unfreezing Pass (12 Epochs)...
⚠️ Operating with Layer-Wise Learning Rates to break the 5% EER ceiling.
--------------------------------------------------------------------------------
🚀 Deep Epoch 01/12 | Train Loss: 11.8906 | Val EER: 6.67%
🚀 Deep Epoch 02/12 | Train Loss: 7.6403 | Val EER: 5.00%
🚀 Deep Epoch 03/12 | Train Loss: 6.7004 | Val EER: 3.33%
🚀 Deep Epoch 04/12 | Train Loss: 3.8692 | Val EER: 3.33%
🚀 Deep Epoch 05/12 | Train Loss: 3.8154 | Val EER: 2.22%
🚀 Deep Epoch 06/12 | Train Loss: 3.8651 | Val EER: 2.78%
🚀 Deep Epoch 07/12 | Train Loss: 2.3933 | Val EER: 1.67%
🚀 Deep Epoch 08/12 | Train Loss: 2.5045 | Val EER: 2.22%
🚀 Deep Epoch 09/12 | Train Loss: 1.6810 | Val EER: 2.22%
🚀 Deep Epoch 10/12 | Train Loss: 2.2284 | Val EER: 2.22%
🚀 Deep Epoch 11/12 | Train Loss: 1.8157 | Val EER: 2.22%
🚀 Deep Epoch 12/12 | Train Loss: 2.0405 | Val EER: 2.22%

🏁 Loading peak performance Elite check

In [8]:
TARGET_FOLDER = "/kaggle/input/datasets/animeshshukla20/more-ai-generated-voice"
ELITE_THRESHOLD = 0.2611 # The optimal threshold calculated from your V7 Elite run

print("Loading V7 Elite peak performance weights...")
# Reusing the existing model instance configuration
model.load_state_dict(torch.load("telephony_forensics_v7_elite.pth", map_location=DEVICE))
model.eval()

# Setting up the uniform telephony simulation filter
telephony_filter = nn.Sequential(T.Resample(16000, 8000), T.Resample(8000, 16000)).to(DEVICE)

files = [f for f in os.listdir(TARGET_FOLDER) if f.lower().endswith(('.mp3', '.wav'))]
print(f"\n🕵️ Running Final Production Audit on '{os.path.basename(TARGET_FOLDER)}' ({len(files)} files)...")
print(f"{'FILENAME':<45} | {'TELEPHONY SCORE':<18} | {'FINAL VERDICT'}")
print("-" * 80)

intercepted_fakes = 0
total_audited = 0

with torch.no_grad():
    for f in files:
        full_path = os.path.join(TARGET_FOLDER, f)
        try:
            # Reconstruct production preprocessing stringently
            wav, sr = torchaudio.load(full_path)
            if wav.shape[0] > 1: wav = torch.mean(wav, dim=0, keepdim=True)
            if sr != 16000: wav = T.Resample(sr, 16000)(wav)
            
            # Evaluate under simulated telephone connection constraints
            wav = telephony_filter(wav.to(DEVICE))
            if wav.shape[1] > 64000: wav = wav[:, :64000]
            else: wav = torch.nn.functional.pad(wav, (0, 64000 - wav.shape[1]))
            
            prob = torch.sigmoid(model(wav.unsqueeze(0))).item()
            total_audited += 1
            
            is_fake = prob < ELITE_THRESHOLD
            verdict = "🚨 DEEPFAKE" if is_fake else "✅ CLEAN HUMAN"
            if is_fake: intercepted_fakes += 1
                
            prob_str = f"{prob:.4f}"
            print(f"{f[:43]:<45} | {prob_str:<18} | {verdict}")
        except Exception as e:
            print(f"❌ Error on {f}: {e}")

print("-" * 80)
final_detection_rate = (intercepted_fakes / total_audited) * 100 if total_audited > 0 else 0
print(f"🏆 Final Blind Intercept Profile: {intercepted_fakes}/{total_audited} caught ({final_detection_rate:.2%}).")

Loading V7 Elite peak performance weights...

🕵️ Running Final Production Audit on 'more-ai-generated-voice' (35 files)...
FILENAME                                      | TELEPHONY SCORE    | FINAL VERDICT
--------------------------------------------------------------------------------
15.mp3                                        | 0.0019             | 🚨 DEEPFAKE
voice_preview_matt - dramatic radio podcast   | 0.0003             | 🚨 DEEPFAKE
4.mp3                                         | 0.9372             | ✅ CLEAN HUMAN
10.mp3                                        | 0.0005             | 🚨 DEEPFAKE
3.mp3                                         | 0.0004             | 🚨 DEEPFAKE
lottery.mp3                                   | 0.0031             | 🚨 DEEPFAKE
25.mp3                                        | 0.9738             | ✅ CLEAN HUMAN
breakup.mp3                                   | 0.0012             | 🚨 DEEPFAKE
17.mp3                                        | 0.0322             

In [11]:
# --- STEP: HIGH-SECURITY CONSTRAINT AUDIT (CORRECTED PRINT SYNTAX) ---
import os
import torch
import torchaudio
import torchaudio.transforms as T

TARGET_FOLDER = "/kaggle/input/datasets/animeshshukla20/more-ai-generated-voice"
# Tightening the gate: Anything less than 85% certain to be human is flagged as a fake
SECURITY_THRESHOLD = 0.8500 

print(f"🔒 Hardening Security Gatekeeper to Threshold: {SECURITY_THRESHOLD}")
print(f"{'FILENAME':<45} | {'SCORE':<10} | {'VERDICT'}")
print("-" * 75)

model.eval()
telephony_filter = torch.nn.Sequential(T.Resample(16000, 8000), T.Resample(8000, 16000)).to(DEVICE)

intercepted = 0
total = 0

with torch.no_grad():
    for f in os.listdir(TARGET_FOLDER):
        if not f.lower().endswith(('.mp3', '.wav')): continue
        full_path = os.path.join(TARGET_FOLDER, f)
        
        wav, sr = torchaudio.load(full_path)
        if wav.shape[0] > 1: wav = torch.mean(wav, dim=0, keepdim=True)
        if sr != 16000: wav = T.Resample(sr, 16000)(wav)
        wav = telephony_filter(wav.to(DEVICE))
        if wav.shape[1] > 64000: wav = wav[:, :64000]
        else: wav = torch.nn.functional.pad(wav, (0, 64000 - wav.shape[1]))
        
        prob = torch.sigmoid(model(wav.unsqueeze(0))).item()
        total += 1
        
        # Apply strict security boundary
        is_fake = prob < SECURITY_THRESHOLD
        if is_fake: intercepted += 1
        verdict = "🚨 DEEPFAKE" if is_fake else "✅ HUMAN"
        
        # Fixed formatting: Convert float to string first, then pad the column layout
        score_str = f"{prob:.4f}"
        print(f"{f[:43]:<45} | {score_str:<10} | {verdict}")

print("-" * 75)
print(f"🎯 Strict Security Summary: Caught {intercepted}/{total} fakes ({intercepted/total:.2%}).")

🔒 Hardening Security Gatekeeper to Threshold: 0.85
FILENAME                                      | SCORE      | VERDICT
---------------------------------------------------------------------------
15.mp3                                        | 0.0019     | 🚨 DEEPFAKE
voice_preview_matt - dramatic radio podcast   | 0.0003     | 🚨 DEEPFAKE
4.mp3                                         | 0.9372     | ✅ HUMAN
10.mp3                                        | 0.0005     | 🚨 DEEPFAKE
3.mp3                                         | 0.0004     | 🚨 DEEPFAKE
lottery.mp3                                   | 0.0031     | 🚨 DEEPFAKE
25.mp3                                        | 0.9738     | ✅ HUMAN
breakup.mp3                                   | 0.0012     | 🚨 DEEPFAKE
17.mp3                                        | 0.0322     | 🚨 DEEPFAKE
originalvo-medieval-gamer-voice-donx27t-for   | 0.0028     | 🚨 DEEPFAKE
Hello and welcome Th.mp3                      | 0.0009     | 🚨 DEEPFAKE
19.mp3            

In [12]:
MODEL_PATH = "/kaggle/working/telephony_forensics_v7_elite.pth"
WILD_REAL_DIR = "/kaggle/input/datasets/abdallamohamed312/in-the-wild-audio-deepfake/release_in_the_wild/real"
VOXCELEB_ROOT = "/kaggle/input/datasets/tnginhnht/voxceleb1/vox1_dev_wav/wav"

# Operating under the strict high-security threshold
SECURITY_THRESHOLD = 0.8500 

print(f"🏗️ Loading V7 Elite weights into evaluation stream...")
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# 2. TELEPHONY PREPROCESSING FILTER
telephony_filter = nn.Sequential(T.Resample(16000, 8000), T.Resample(8000, 16000)).to(DEVICE)

def extract_real_tensor(path):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1: wav = torch.mean(wav, dim=0, keepdim=True)
    if sr != 16000: wav = T.Resample(sr, 16000)(wav)
    wav = telephony_filter(wav.to(DEVICE))
    if wav.shape[1] > 64000: wav = wav[:, :64000]
    else: wav = torch.nn.functional.pad(wav, (0, 64000 - wav.shape[1]))
    return wav.unsqueeze(0)

# 3. GATHERING 600 RANDOM SAMPLES
random.seed(42)

print("🔍 Crawling In-The-Wild real voice directories...")
wild_files = [os.path.join(WILD_REAL_DIR, f) for f in os.listdir(WILD_REAL_DIR) if f.lower().endswith(('.mp3', '.wav'))]
random.shuffle(wild_files)
selected_wild = wild_files[:300]

print("🔍 Crawling VoxCeleb1 nested speaker folders...")
vox_files = glob(os.path.join(VOXCELEB_ROOT, "**/*.wav"), recursive=True)
random.shuffle(vox_files)
selected_vox = vox_files[:300]

# Combine to create a 600-file true human validation pool
expanded_real_pool = selected_wild + selected_vox
print(f"📁 Balanced Human Test Pool compiled: {len(expanded_real_pool)} tracks ready.\n")

# 4. EXECUTE AUDIT
cleared_humans = 0
false_alarms = 0
total_processed = 0

print(f"🕵️ Running Integrity Check with Security Gate: {SECURITY_THRESHOLD}")
print(f"{'SOURCE / FILENAME':<50} | {'RAW SCORE':<12} | {'SECURITY STATUS'}")
print("-" * 85)

with torch.no_grad():
    for path in expanded_real_pool:
        try:
            tensor = extract_real_tensor(path)
            prob = torch.sigmoid(model(tensor)).item()
            total_processed += 1
            
            # Evaluate using the strict security rule
            is_human_passed = prob >= SECURITY_THRESHOLD
            
            if is_human_passed:
                cleared_humans += 1
                status = "✅ CLEAR HUMAN"
            else:
                false_alarms += 1
                status = "🚨 FALSE ALARM (AI)"
                
            # Print snapshot log of the first 15 entries to observe score variations
            if total_processed <= 15:
                src = "VoxCeleb" if "vox1_dev_wav" in path else "In-Wild"
                fn = os.path.basename(path)
                display_str = f"[{src}] {fn}"
                score_str = f"{prob:.4f}"
                print(f"{display_str[:48]:<50} | {score_str:<12} | {status}")
            elif total_processed == 16:
                print("... [Processing the remaining real voice profiles across the pipeline] ...")
                
        except Exception as e:
            pass # Gracefully pass files with unreadable sample layouts

# 5. METRICS REPORT
clearance_rate = (cleared_humans / total_processed) * 100 if total_processed > 0 else 0
print("\n📊 ==================================================")
# Renamed from V6 to V7 to match current iteration
print("        V7 ELITE HIGH-SECURITY SANITY PROFILE         ")
print("==================================================")
print(f"✓ Total Real Audio Files Tested: {total_processed}")
print(f"🔒 Correctly Cleared Humans:     {cleared_humans} / {total_processed}")
print(f"⚠️ Triggered False Alarms:       {false_alarms} / {total_processed}")
print(f"--------------------------------------------------")
print(f"🎯 Real Voice User Clearance Rate: {clearance_rate:.2%}")
print(f"🎛️ Active Hardened Threshold:       {SECURITY_THRESHOLD:.4f}")
print("==================================================\n")

🏗️ Loading V7 Elite weights into evaluation stream...
🔍 Crawling In-The-Wild real voice directories...
🔍 Crawling VoxCeleb1 nested speaker folders...
📁 Balanced Human Test Pool compiled: 600 tracks ready.

🕵️ Running Integrity Check with Security Gate: 0.85
SOURCE / FILENAME                                  | RAW SCORE    | SECURITY STATUS
-------------------------------------------------------------------------------------
[In-Wild] 5353.wav                                 | 1.0000       | ✅ CLEAR HUMAN
[In-Wild] 13565.wav                                | 0.9999       | ✅ CLEAR HUMAN
[In-Wild] 29319.wav                                | 0.9999       | ✅ CLEAR HUMAN
[In-Wild] 19778.wav                                | 1.0000       | ✅ CLEAR HUMAN
[In-Wild] 22376.wav                                | 0.9989       | ✅ CLEAR HUMAN
[In-Wild] 25464.wav                                | 1.0000       | ✅ CLEAR HUMAN
[In-Wild] 14780.wav                                | 0.9952       | ✅ CLEAR HUMA